# Data

We have prepared a 90 second snippet of raw data from a much longer openly available dataset. The 90s snippet is called `dandiset-000939-90s.zip` on this Zenodo page: https://zenodo.org/records/23373455 If you'd like to follow along, please download it, unzip it and put the data somewhere. I'll save it in a folder called `data` in this repository.

The dataset we will use through the majority of this course is [hosted on DANDI](https://dandiarchive.org/dandiset/000939?search=000939). The data was used in [Duszkiewicz et al., 2024](https://www.nature.com/articles/s41593-024-01588-5). The experiments recorded from the postsubiculum, a parahippocampal cortical region of the parahippocampal formation and a key hub of the spatial navigation system, using a 64-channel single-shank Cambridge Neurotech probe. They recorded from mice using open-field exploration (as well as during sleep) in which they measured the head-direction of the mice. They were interested in investigating neural tuning for head direction (i.e. head-direction tuned neurons fire when the head of an animal is facing a particular direction or angle).



# Recordings

We'll use SpikeInterface to read in our data. This will create a `Recording` object that we can play with.

Our data was saved by SpikeInterface, so we'll just use the generic `load` function, but SpikeInterface supports many formats, see a full list here: https://spikeinterface.readthedocs.io/en/stable/modules/extractors.html#raw-data-formats

For example, if your data is from an OpenEphys acquisition system, you can use `si.read_openephys` or if it is from a SpikeGLX system, you can use `si.read_spikeglx`.

In [ ]:
import spikeinterface.full as si
from pathlib import Path
import os

In [ ]:
# change these if working on your laptop
base_data_folder = Path("/Users/christopherhalcrow/Work/fromgit/SpikeInterface-Training-OpenLabOnline-Oct26/data")

In [ ]:
recording_name = 'dandiset-000939-90s'

path_to_recording = base_data_folder / recording_name

recording = si.load(path_to_recording)

The recording is an object, and we can find out information about it like so

In [ ]:
recording

You can also directly access a lot of this information from the recording object:

In [ ]:
sampling_frequency = recording.sampling_frequency
channel_locations = recording.get_channel_locations()
channel_ids = recording.channel_ids

print(channel_ids)

Some information is saved as recording "annotations". Let's see which annotations are saved

In [ ]:
recording.get_annotation_keys()

We can also add our own annotations

In [ ]:
recording.set_annotation('experimenter_name', 'Adrian Duszkiewicz')

Most importantly, the recording contains information about the geometric layout of the probe. We can take a look using the `plot_probe_map` plotting function.

In [ ]:
%matplotlib widget
si.plot_probe_map(recording)

The probe layout is essential for the spike sorting algorithms. Hence, it's very important that the probe information is correct.

Your actual raw data is (usually) saved as a binary (or "raw" or "dat") file. This is just a 2D array, with time on one axis and channel index on the other. A good way to think of the probe is that it links the geometric positions of each electrode to the row in your 2D array which contains the raw signal from that channel. If you've done experiments, you know that this "wiring" can depends on lots of intermediate information: how the head stage is wired, a channel selection on your acquisition system etc.

We can plot the raw traces. Note that not all data gets loaded: only the data in the `time_range` is loaded.

In [ ]:
si.plot_traces(
    recording,
    mode="line",  # plot the traces as lines, rather than a heatmap
    time_range = (8,10) # plot all data from 8-10 seconds
)

Hmmm... that doesn't look helpful. We can _slice_ our recording to focus on a small bit of it; either in time or in channels.

In [ ]:
time_sliced_recording = recording.time_slice(start_time=0, end_time=10)
channel_sliced_recording = time_sliced_recording.select_channels(channel_ids=[0,1,2,3])
# Note: you can also remove chanels using `recording.remove_channels(...)

si.plot_traces(channel_sliced_recording)

There are also some interactive plotting tools within SpikeInterface that you can use when in a Jupyter notebook. We need to tell Jupyter to use these tools using a "magic command":

In [ ]:
%matplotlib widget
si.plot_traces(recording, backend='ipywidgets')

# Generating your own Probe

The data we've loaded is great: the probe is auto-generated when you load in the data. This is also generally true for openephys and spikeglx data: you shouldn't have to touch anything to get all of the probe geometry "for free".

But that's not always the case.

Suppose you have a custom probe with 4-shanks, with 8 channels on each shank. We can generate some artificial data to imitate this scenario

In [ ]:
fake_recording = si.generate_recording(
    num_channels = 4*8,
    durations = [300],
    set_probe=False,
)

print(fake_recording.has_probe())

We can make a probe by creating a `Probe` object from `ProbeInterface`. At a minimum, this needs the locations of each channel. Read more in the ProbeInterface docs (https://probeinterface.readthedocs.io/en/stable/index.html)

In [ ]:
import numpy as np

positions_on_one_shank = np.array([
    [0,0],
    [20,0],
    [0,20],
    [20,20],
    [0,40],
    [20,40],
    [0,60],
    [20,60],
])

all_positions = np.concatenate([
    positions_on_one_shank, 
    positions_on_one_shank+[100,0], 
    positions_on_one_shank+[200,0], 
    positions_on_one_shank+[300,0]
])

In [ ]:
from probeinterface import Probe
probe = Probe(ndim=2, si_units='um')
probe.set_contacts(positions=all_positions)

# We need to "wire" the probe, to say which channel corresponds to which column in the raw traces file. 
probe.set_device_channel_indices(range(0,4*8))

# And we'll make an auto-shape in the hope of making prettier plots
probe.create_auto_shape()

fake_recording.set_probe(probe)

In [ ]:
si.plot_probe_map(fake_recording)

These four shanks are far apart from each other, and we might want to deal with them independently. We can set a "group" property to each electrode:

In [ ]:
fake_recording.set_channel_groups([0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3])

We can then split the recording using the `split_by` method:

In [ ]:
grouped_recordings = fake_recording.split_by('group')
grouped_recordings

As we'll see later, many of `SpikeInterface`'s functions work exactly the same way with a recording, or a group of recordings (like the one above)

`ProbeInterface` also has some helper probe creation functions such as `generate_multi_shank` and `generate_linear`. Read more in the docs: https://probeinterface.readthedocs.io/en/stable/index.html

There is also a library of probes you can load in (https://probeinterface.readthedocs.io/en/stable/examples/ex_10_get_probe_from_library.html), mostly from Cambridge Neurotech and NeuroNexus.

`SpikeInterface` also contains some pretty advanced generation tools. We can make fake data that imitates a multi-shank recording like so

In [ ]:
from probeinterface import generate_multi_shank
two_shank_probe = generate_multi_shank(num_shank=2, num_contact_per_column=8, num_columns=2)
two_shank_probe.set_device_channel_indices(range(32))
si_generated_recording, _ = si.generate_ground_truth_recording(
    durations=[300],
    num_channels=32,
    probe=two_shank_probe
)
si.plot_probe_map(si_generated_recording)

# Compression

`SpikeInterface` allows you to compress your raw data if you save it in the [Zarr](https://zarr.dev/) format. Read more [here](https://iopscience.iop.org/article/10.1088/1741-2552/acf5a4/meta)!

We just need to adjust our settings when we save the recording

In [ ]:
preprocessed_zarr_folder = base_data_folder / "raw_recording.zarr"
recording.save(folder=preprocessed_zarr_folder, format="zarr")

Let's check if this is smaller than our binary recording

In [ ]:
def get_folder_size_mb(path="."):
    total_size = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 ** 2)


size_binary = get_folder_size_mb(path=path_to_recording)
size_zarr = get_folder_size_mb(path=preprocessed_zarr_folder)
print(f"Size of preprocessed binary folder: {size_binary:.2f} MB")
print(f"Size of preprocessed Zarr folder: {size_zarr:.2f} MB")

You can use different compression strategies to try and make your file even smaller. Read more in our docs: https://spikeinterface.readthedocs.io/en/stable/modules/core.html#saving-loading-and-compression

Be warned that if you use a codec (like WavPak) to compress your file, whoever opens the file will need to install the codec too.